In [4]:
import ollama
import json
import re
import os
from datetime import datetime
from tavily import TavilyClient
from dotenv import load_dotenv

# ─────────────────────────────────────────────
# Load Environment Variables
# ─────────────────────────────────────────────
load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
print(f"Using Ollama model : {OLLAMA_MODEL}")
print(f"Tavily key configured: {'Yes' if TAVILY_API_KEY else 'NO – please set it'}")

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

Using Ollama model : llama3.2:3b
Tavily key configured: Yes


In [8]:
import ollama

generation_chat_history = [
    {
        "role": "user",
        "content": "Write Python code for merge sort"
    }
]

response = ollama.chat(
    model="llama3.2:3b",
    messages=generation_chat_history
)

mergesort_code = response["message"]["content"]

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

print(mergesort_code)

# Merge Sort Algorithm in Python

Merge sort is a divide-and-conquer algorithm that splits an input array into two halves, recursively sorts each half, and then merges the sorted halves.

## Code
```python
def merge_sort(arr):
    """
    Sorts an input array using the merge sort algorithm.

    Args:
        arr (list): The input array to be sorted.

    Returns:
        list: The sorted array.
    """

    # Base case: If the array has 1 or fewer elements, it is already sorted
    if len(arr) <= 1:
        return arr

    # Find the middle index of the array
    mid = len(arr) // 2

    # Recursively sort the left and right halves of the array
    left_half = merge_sort(arr[:mid])
    right_half = merge_sort(arr[mid:])

    # Merge the sorted left and right halves
    return merge(left_half, right_half)


def merge(left, right):
    """
    Merges two sorted arrays into a single sorted array.

    Args:
        left (list): The first sorted array.
        right (list): The second sor

In [9]:
class ResearchAgent:
    """
    A ReAct agent that plans research with an Ollama LLM,
    gathers information via Tavily web search, and compiles
    a structured markdown report.
    """

    def __init__(self, topic: str, ollama_model: str = OLLAMA_MODEL,
                 tavily_api_key: str = TAVILY_API_KEY):
        self.topic          = topic
        self.ollama_model   = ollama_model
        self.tavily         = TavilyClient(api_key=tavily_api_key)
        self.questions      = []          # populated in planning phase
        self.research_data  = {}          # {question: [{title, content}, ...]}

    # ─────────────────────────────────────────
    # PHASE 1 – REASON: generate research questions
    # ─────────────────────────────────────────
    def generate_questions(self) -> list[str]:
        """
        Asks the Ollama LLM to decompose the topic into
        5–6 specific research questions.

        The LLM is instructed to return ONLY a JSON array so
        the response is easy to parse programmatically.
        """
        print(f"\n[REASON] Generating research questions for: '{self.topic}'")

        prompt = f"""You are a research planning assistant.
Given the topic: "{self.topic}"

Generate exactly 5 well-structured research questions that cover different
important aspects of this topic.

Rules:
- Return ONLY a valid JSON array of strings, nothing else.
- No preamble, no explanation, no markdown fences.
- Each question should be specific and searchable.

Example format:
["Question 1?", "Question 2?", "Question 3?", "Question 4?", "Question 5?"]"""

        response = ollama.chat(
            model=self.ollama_model,
            messages=[{"role": "user", "content": prompt}]
        )

        raw_text = response["message"]["content"].strip()

        # Robustly extract a JSON array even if the model adds extra text
        json_match = re.search(r'\[.*?\]', raw_text, re.DOTALL)
        if json_match:
            self.questions = json.loads(json_match.group())
        else:
            # Fallback: split line-by-line
            lines = [l.strip().lstrip('0123456789.-) ') for l in raw_text.splitlines()
                     if l.strip() and '?' in l]
            self.questions = lines[:6]

        print(f"  ✓ Generated {len(self.questions)} questions:")
        for i, q in enumerate(self.questions, 1):
            print(f"    {i}. {q}")

        return self.questions

    # ─────────────────────────────────────────
    # PHASE 2 – ACT: web search for each question
    # ─────────────────────────────────────────
    def search_web(self, max_results: int = 3) -> dict:
        """
        For each research question, performs a Tavily web search
        and extracts the title and content of every result.

        Args:
            max_results: number of search results to retrieve per question

        Returns:
            dict mapping each question to a list of {title, content} dicts
        """
        if not self.questions:
            raise RuntimeError("Run generate_questions() first.")

        print("\n[ACT] Searching the web for each question...")

        for question in self.questions:
            print(f"\n  Searching: {question}")
            try:
                response = self.tavily.search(
                    query=question,
                    max_results=max_results,
                    search_depth="advanced"      # deeper, higher-quality results
                )

                results = []
                for r in response.get("results", []):
                    results.append({
                        "title"  : r.get("title", "No title"),
                        "url"    : r.get("url", ""),
                        "content": r.get("content", "No content available")[:800]  # trim
                    })

                self.research_data[question] = results
                print(f"    ✓ Found {len(results)} result(s)")

            except Exception as e:
                print(f"    ✗ Search failed: {e}")
                self.research_data[question] = []

        return self.research_data

    # ─────────────────────────────────────────
    # PHASE 3 – REPORT: compile structured report
    # ─────────────────────────────────────────
    def compile_report(self) -> str:
        """
        Formats all collected research data into a structured
        markdown report with title, introduction, per-question
        sections, and a conclusion.
        """
        if not self.research_data:
            raise RuntimeError("Run search_web() first.")

        print("\n[REPORT] Compiling final report...")

        date_str = datetime.now().strftime("%B %d, %Y")
        lines = []

        # ── Title & Introduction ──────────────────────────
        lines.append(f"# Research Report: {self.topic}")
        lines.append(f"\n*Generated on {date_str} by ReAct Research Agent (Ollama + Tavily)*")
        lines.append("\n---")
        lines.append("\n## Introduction")
        lines.append(
            f"This report presents a comprehensive overview of **{self.topic}** "
            f"compiled by an AI research agent. The agent autonomously generated "
            f"{len(self.questions)} targeted research questions and retrieved "
            f"up-to-date information from the web using the Tavily search API."
        )

        # ── One section per research question ────────────
        lines.append("\n---")
        lines.append("\n## Research Findings")

        for i, (question, results) in enumerate(self.research_data.items(), 1):
            lines.append(f"\n### {i}. {question}")

            if not results:
                lines.append("*No results found for this question.*")
                continue

            for r in results:
                lines.append(f"\n**{r['title']}**")
                lines.append(f"*Source: {r['url']}*")
                lines.append(f"\n{r['content']}")
                lines.append("")

        # ── Conclusion ────────────────────────────────────
        lines.append("\n---")
        lines.append("\n## Conclusion")
        lines.append(
            f"This report covered {len(self.questions)} key aspects of **{self.topic}** "
            f"through automated web research. The findings above represent current, "
            f"web-sourced information gathered by the ReAct agent. For deeper analysis, "
            f"the sources linked in each section can be explored further."
        )
        lines.append("\n---")
        lines.append("\n*Report generated by ReAct Research Agent | Powered by Ollama (LLM) & Tavily (Search)*")

        report = "\n".join(lines)
        print("  ✓ Report compiled successfully.")
        return report

    # ─────────────────────────────────────────
    # Convenience: run the full pipeline
    # ─────────────────────────────────────────
    def run(self, max_results: int = 3) -> str:
        """
        Executes the full ReAct pipeline:
          1. generate_questions  (Reason)
          2. search_web          (Act)
          3. compile_report      (Observe + Report)
        """
        self.generate_questions()
        self.search_web(max_results=max_results)
        return self.compile_report()

In [10]:
# ─── Set your research topic here ───────────────────────
TOPIC = "Climate Change"
# ────────────────────────────────────────────────────────

agent  = ResearchAgent(topic=TOPIC)
report = agent.run(max_results=3)

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)


[REASON] Generating research questions for: 'Climate Change'
  ✓ Generated 5 questions:
    1. How do climate change mitigation strategies impact local economies?
    2. Can machine learning algorithms accurately predict ocean acidification rates?
    3. What is the relationship between deforestation and global temperature regulation?
    4. How do climate change projections affect global food security and agricultural productivity?
    5. Are urban green spaces effective in reducing heat island effects and improving air quality?

[ACT] Searching the web for each question...

  Searching: How do climate change mitigation strategies impact local economies?
    ✓ Found 3 result(s)

  Searching: Can machine learning algorithms accurately predict ocean acidification rates?
    ✓ Found 3 result(s)

  Searching: What is the relationship between deforestation and global temperature regulation?
    ✓ Found 3 result(s)

  Searching: How do climate change projections affect global food security

In [11]:
# Render the report as formatted markdown inside the notebook
from IPython.display import Markdown, display
display(Markdown(report))

# Research Report: Climate Change

*Generated on March 16, 2026 by ReAct Research Agent (Ollama + Tavily)*

---

## Introduction
This report presents a comprehensive overview of **Climate Change** compiled by an AI research agent. The agent autonomously generated 5 targeted research questions and retrieved up-to-date information from the web using the Tavily search API.

---

## Research Findings

### 1. How do climate change mitigation strategies impact local economies?

**Policy challenge: Local economies and net-zero - What Works Growth**
*Source: https://whatworksgrowth.org/resource-library/policy-challenge-local-economies-and-net-zero/*

Policymakers around the world are aiming to meet the carbon emissions targets set out in the 2015 Paris Agreement. Meeting these targets for mitigation of climate change requires transformation of economies through building capacity needed by new green sectors, like zero carbon energy production, electric vehicle production, and carbon capture and storage, and helping existing sectors to reduce their carbon emissions. Adaptation to climate change also reshapes economies, for example through investments in flood defence technologies and infrastructure or heat resilient building design and urban planning. [...] Local economic policy needs to consider the impact of mitigation and adaption on local economic outcomes and the potential economic complications of transition, such as job losses in 


**[PDF] Climate change mitigation strategies: impacts and obstacles in low**
*Source: https://www.clubofrome.org/wp-content/uploads/2022/07/Earth4All_Deep_Dive_Ghosh2.pdf*

reliant on renewable energy expands, shows how mining can adversely affect local populations who are mostly not adequately compensated for their land and livelihood losses or for the ecological impact. Similarly, the examples provided by waste recycling have pointed to the neo-colonial patterns in the global waste trade, most of all in plastics but also in other materials. Finally, the recent trends in climate finance show how existing efforts to make available the necessary finance for climate mitigation, adaptation, and loss and damage, are simply not being provided to most of the world, even as public subsidies and private finance for fossil fuels are massive in relation to the small amounts of climate finance. [...] Climate change mitigation strategies: impacts and obstacles in low- an


**How climate change mitigation makes economic sense**
*Source: https://climateactiontracker.org/publications/how-climate-change-mitigation-makes-economic-sense/*

Governments can offset the cost of stronger climate policies by taking into account the savings associated with reduced mortality from harmful anthropological air pollutants such as particulate matter and ozone.
 Reduced air pollution lowers the risk of mortality from air pollution-related illnesses, such as respiratory and cardiovascular diseases, that would otherwise impose significant economic impacts on national health care systems and economies. [...] We also exclude the actual costs – human, economic, environmental, and social - of climate change, such as those due to sea level rise, extreme weather events, reduced crop yields, and the need for adaptation. If they were taken into consideration, the cost-effectiveness of mitigation in many regions would likely become even more attract


### 2. Can machine learning algorithms accurately predict ocean acidification rates?

**Hybrid machine learning algorithms accurately predict marine ...**
*Source: https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2025.1458014/full*

et al., 2021). As such, while accurate predictions are needed, it is also important to extract the model features and the interactions within environmental data (Murdoch et al., 2019). It is based on the response of biological data and interactions between environmental variables that the major oceanographic processes can be understood and monitored. [...] 4 was significantly higher than that of Associations 1, 5, and 6. [...] ### 2.6 Supervised upscaling


**pH acidification in the Red Sea: a machine learning-based ...**
*Source: https://news-oceanacidification-icc.org/2025/08/19/ph-acidification-in-the-red-sea-a-machine-learning-based-validation-study/*

mixing of waters, and CO₂ infusion from the atmosphere accurately. Therefore, this research proposes a comprehensive approach for evaluating long-term changes in pH levels using robust data, improving strategic environmental governance in marine ecosystems. ML-based algorithms offer more integrated, cost-effective, and scalable solutions for monitoring ocean acidification, outperforming traditional approaches in both efficiency and adaptability. [...] This study presents application and performance comparison of various machine learning (ML) techniques to analyze pH variations in the Red Sea between the years 2021 and 2024, utilizing satellite remote sensing from the Copernicus Programme. The accuracy of the model is enhanced by employing data preprocessing. The performance of a number of 


**[PDF] Crafting the Future: Machine learning for ocean forecasting - Reports**
*Source: https://sp.copernicus.org/articles/5-opsr/22/2025/sp-5-opsr-22-2025.pdf*

Within the realm of machine learning (ML) applications for ocean forecasting, progress has been somewhat limited.
Recent developments have marked a shift in this landscape, particularly with the introduction of Fourier neural opera-tors for modeling oceanic processes, as suggested by Bire et al. (2023), Chattopadhyay et al. (2024), and Sun et al. (2024). [...] Chapter 8.3 – Ocean prediction: present status and state of the art Crafting the Future: Machine learning for ocean forecasting Patrick Heimbach1, Fearghal O’Donncha2, Timothy A. Smith3, Jose Maria Garcia-Valdecasas4, Alain Arnaud5, and Liying Wan6 1Oden Institute for Computational Engineering and Sciences, The University of Texas at Austin, Austin, TX, United States 2IBM Research, Dublin, Ireland 3NOAA Physical Sciences Laboratory, 


### 3. What is the relationship between deforestation and global temperature regulation?

**Deforestation Creates Extreme Heat in the Tropics**
*Source: https://www.wri.org/insights/deforestation-heat-stress-tropics*

In a 2021 study focusing on two states in Brazil, Mato Grosso and Pará, where more than half of Brazilian deforestation occurred between 2008 and 2019, researchers found a strong relationship between large-scale deforestation and lost working hours. In deforested areas, 45% of workers lost a half hour or more of daily safe work time versus less than 5% of workers in forested areas.

Job and income loss will translate to additional negative impacts on human health and, for those who are employed, may mean added pressure to work in hazardous conditions. The more tropical forests are kept intact, the less severe the inevitable economic impacts of global temperature rise will be for tropical countries. [...] Not only do rising temperatures in the tropics endanger human health; they can also ha


**Global forestation and deforestation affect remote climate via ... - PMC**
*Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC9532392/*

Here we find that forestation leads to global warming of +0.5 K, which is most pronounced over northern extratropical land. Consequently, the meridional heat transport in the Northern Hemisphere decreases in the forestation simulation. The reduction manifests itself predominantly in a weakened Atlantic meridional overturning circulation (AMOC). Warming of high-latitude land surfaces results further in weaker and poleward-shifted weather systems, which, via momentum feedback to the mean flow, leads to attenuation and poleward displacement of the midlatitude westerlies. Deforestation leads to the global cooling of −1.6 K, a stronger AMOC, an accelerated extratropical jet stream, a southward shift of the intertropical convergence zone and a stronger Hadley cell in boreal winter, and a weaker 


**What is the Relationship Between Deforestation And Climate Change?**
*Source: https://www.rainforest-alliance.org/insights/what-is-the-relationship-between-deforestation-and-climate-change/*

Find products with the Rainforest Alliance Certified seal available near you!

Find products with the Rainforest Alliance Certified seal available near you!

Home / Insights / What is the Relationship Between Deforestation And Climate Change?

# What is the Relationship Between Deforestation And Climate Change?

Filed Under: Insights  |  Tagged: Deforestation, Climate   
 Last updated August 12, 2018

deforestation and climate change are deeply linked. trees are our greatest ally in fighting global warming.

What, exactly, is the relationship between deforestation and climate change? The Rainforest Alliance breaks down the numbers for you—and explains our innovative approach to keeping forests standing. [...] When we clear forests, we’re not only knocking out our best ally in capturing the


### 4. How do climate change projections affect global food security and agricultural productivity?

**How Climate Change is Impacting Global Food Systems ...**
*Source: https://sowhatelse.org/how-climate-change-is-impacting-global-food-systems-and-security/*

Agriculture is both a victim and a contributor to climate change. While it is directly impacted by extreme weather patterns, it is also responsible for significant greenhouse gas emissions. Understanding the relationship between agriculture and climate change is key to addressing food security.

 Carbon footprint of agriculture: Large-scale farming practices contribute to greenhouse gas emissions, accelerating climate change.
 Impact of climate change on crop yields: Changes in temperature and rainfall affect the productivity of staple crops like wheat, rice, and corn.
 Soil degradation: Unsustainable farming practices, combined with climate change, lead to soil erosion, reducing the land’s ability to produce food. [...] ## The Connection Between Climate Change and Food Systems

Climate ch


**Impacts of climate change on global agriculture accounting ... - Nature**
*Source: https://www.nature.com/articles/s41586-025-09085-w*

A key finding is that global populations exhibit extensive adaptation to climate already, especially in relatively low income and hot regions of the world—with the exception of the world’s poorest, who depend heavily on cassava and face higher potential losses (Fig. 3b,c, Supplementary Information, section I.3 and Supplementary Fig. 9). We also observe that breadbaskets of the world, in which the climate is moderate, exhibit more limited adaptation at present. Because such a large fraction of agricultural production is concentrated in these wealthy-but-low-adaption regions, they dominate projections of global calorie production, generating much of the global food security risk we document. This result is consistent with earlier regional findings that US crop systems are optimized for high 


**Climate Change, Global Food Security, and the U.S. Food System**
*Source: https://www.usda.gov/about-usda/general-information/priorities/climate-solutions/climate-change-global-food-security-and-us-food-system*

Effective beginning 5/20/2025: Please note this site is under review and content may change.

Climate change is likely to diminish continued progress on global food security through production disruptions that lead to local availability limitations and price increases, interrupted transport conduits, and diminished food safety, among other causes.

Climate change can affect food availability, access, utilization, and the stability of each of these over time. Constrictions at any point can lead to food insecurity through the activities of the food system, including food production, transportation, and storage.


### 5. Are urban green spaces effective in reducing heat island effects and improving air quality?

**How can Green Spaces Improve Air Quality? - REACH GREEN**
*Source: https://www.reachgreen.org/2025/04/14/how-can-green-spaces-improve-air-quality/*

Trees and plants act as natural air filters. Through the process of photosynthesis, they absorb carbon dioxide and release oxygen. Additionally, they trap particulate matter (PM), such as dust, dirt, and smoke, on their leaves and bark, reducing the number of harmful particles in the air. According to a study by the U.S. Forest Service, trees in urban areas remove approximately 711,000 metric tons of air pollution annually, providing an estimated $3.8 billion in health benefits.

In addition to filtering pollutants, green spaces help reduce the urban heat island effect, where cities become significantly warmer than surrounding rural areas due to human activities. Cooler temperatures reduce the formation of ground-level ozone, a harmful air pollutant that can cause respiratory issues. [...]


**The Role of Urban Green Spaces in Mitigating the Urban Heat ...**
*Source: https://www.mdpi.com/2071-1050/17/13/6132*

Ventilation is another crucial mechanism that significantly reduces the urban heat island effect. The primary contribution of green spaces to improving ventilation is their role in circulating air and distributing heat in cities. Large open areas, such as parks and green corridors, provide pathways for airflow, creating cooler air pockets and thus diminishing the intensity of heat islands. This was evident in the work of He et al. , where urban planning strategies incorporating ventilation corridors substantially reduced heat stress on a local scale. The study reveals that ventilation corridors have typified the reduction of locally generated heat stress by facilitating airflow along green-lined streets in cities like Tokyo, promoting thermal comfort. Moreover, incorporating water bodies [


**Urban Green Space: Creating a Triple Win for Environmental ... - PMC**
*Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC6888177/*

If well-designed, urban green space—such as street trees, parks, green roofs, and facades—can help achieve reductions in temperature and air pollution in urban areas while simultaneously delivering diverse additional benefits such as biodiverse habitats and enhanced living and recreation areas . Urban heat islands can increase urban temperatures by up to 12 °C compared to non-urban areas. This can exacerbate heat stress in city dwellers . Trees can provide shade and reduce the demand for air conditioning during warm periods , thus reducing energy demand and promoting sustainability. A meta-analysis of the literature on the effect of urban parks on air temperature showed an average cooling effect of approximately 1 °C . This effect exists up to 1 km from the park boundary with factors such 


---

## Conclusion
This report covered 5 key aspects of **Climate Change** through automated web research. The findings above represent current, web-sourced information gathered by the ReAct agent. For deeper analysis, the sources linked in each section can be explored further.

---

*Report generated by ReAct Research Agent | Powered by Ollama (LLM) & Tavily (Search)*

In [12]:
filename = f"report_{TOPIC.replace(' ', '_').lower()}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(report)

print(f"Report saved to: {filename}")

Report saved to: report_climate_change.md
